Célula 1 - Importações e Carga dos Dados da PRF

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações iniciais
sns.set_theme(style="whitegrid")

# 1. Carregar todos os arquivos da PRF do Pará
caminho_prf = '../data/raw/prf/*pa*.csv' 
lista_dfs_prf = [pd.read_csv(arq, sep=',', encoding='latin1', low_memory=False) for arq in glob.glob(caminho_prf)]
df_prf = pd.concat(lista_dfs_prf, ignore_index=True)

# 2. Padronização de colunas
df_prf.columns = df_prf.columns.str.lower().str.replace(' ', '_')
df_prf = df_prf.loc[:, ~df_prf.columns.str.contains('^unnamed')]

print(f"Base da PRF carregada. Formato: {df_prf.shape}")

Base da PRF carregada. Formato: (19664, 37)


Célula 2 - Tratamento de Tipos e Outliers da PRF

In [2]:
# 1. Correção de coordenadas numéricas (Vrgula para Ponto)
for col in ['latitude', 'longitude']:
    if col in df_prf.columns and df_prf[col].dtype == 'object':
        df_prf[col] = df_prf[col].str.replace(',', '.').astype(float)

# 2. Criação de Features Temporais Básicas
if 'data_inversa' in df_prf.columns:
    df_prf['data_inversa'] = pd.to_datetime(df_prf['data_inversa'], format='%Y-%m-%d', errors='coerce')
    df_prf['ano'] = df_prf['data_inversa'].dt.year
    df_prf['mes'] = df_prf['data_inversa'].dt.month

if 'horario' in df_prf.columns:
    df_prf['hora_decimal'] = pd.to_datetime(df_prf['horario'], format='%H:%M:%S', errors='coerce').dt.hour

# 3. Tratamento de Outliers no KM via IQR
df_prf['km'] = df_prf['km'].astype(str).str.replace(',', '.')
df_prf['km'] = pd.to_numeric(df_prf['km'], errors='coerce')
km_valido = df_prf['km'].dropna()

Q1 = km_valido.quantile(0.25)
Q3 = km_valido.quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

# Filtrando a base removendo inconsistências
df_prf_limpo = df_prf[(df_prf['km'].notna()) & (df_prf['km'] >= 0) & (df_prf['km'] <= limite_superior)].copy()

# 4. Remoção de colunas administrativas inuteis para o ML
colunas_inuteis_prf = ['id', 'pesid', 'id_veiculo', 'regional', 'delegacia', 'uop', 'ordem_tipo_acidente']
colunas_drop = [col for col in colunas_inuteis_prf if col in df_prf_limpo.columns]
df_prf_limpo = df_prf_limpo.drop(columns=colunas_drop)

# 5. Exportação do Checkpoint da PRF
os.makedirs('../data/processed', exist_ok=True)
df_prf_limpo.to_csv('../data/processed/prf_limpo.csv', index=False, sep=';', encoding='utf-8')
print(f"Base da PRF exportada com sucesso! Linhas restando: {len(df_prf_limpo)}")

Base da PRF exportada com sucesso! Linhas restando: 18052
